# Qwen-Image-2.1 · Colab 文生图

> ⚠️ **未在 GPU 上实测，首跑需在 Colab 里验证。** 本笔记本按 Qwen-Image-2.1 官方 diffusers 用法编写（`QwenImage21Pipeline`，模型 `Qwen/Qwen-Image-2.1`），参考了 Kaggle 同名笔记本的运行信息（T4 x2，31 分钟跑通）——Kaggle 页面只暴露了运行元信息，未读到源码，故不照搬其实现。
>
> - 运行时：Colab 免费 GPU **T4 16GB**（运行时 → 更改运行时类型 → T4 GPU）。
# T4 16GB 实测：整块搬运会 OOM，改逐层卸载（2026-11-09 Colab 首跑验证）
> - 显存策略：7B DiT（bf16 约 14GB）+ Qwen3-VL 8B 编码器（bf16 约 16GB）塞不进单张 T4，因此默认开 `enable_sequential_cpu_offload()`（CPU 卸载），能跑但比整卡加载慢；首次下载权重约 30GB，需等下载完成。
> - 结果保存：Google Drive `/MyDrive/free-stack/qwen-image-2.1/`（Colab 免费 GPU 只能手动跑，断开即失，产物必须落 Drive）。
> - 许可：Qwen-Image-2.1 采用 **Qwen Research License，仅限非商用**（研究与评估），商用需另向 Qwen 申请授权。

In [ ]:
# ===== 可改参数：只改这里 =====
PROMPT = "A cozy lighthouse on a rocky island at dusk, warm light in the windows, dramatic clouds, cinematic, highly detailed"
NEGATIVE_PROMPT = ""  # 留空即不用负面提示词
WIDTH = 1024          # 建议 1024（约 1MP）；T4 上不要直接上 2048
HEIGHT = 1024
STEPS = 40            # 官方默认 40 步
SEED = 42
OUT_DIR = "/content/drive/MyDrive/free-stack/qwen-image-2.1"

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("输出目录:", OUT_DIR)

In [ ]:
# diffusers 需从 GitHub main 安装（QwenImage21Pipeline 尚未进入已发布的 PyPI 版）
!pip install -q git+https://github.com/huggingface/diffusers transformers accelerate safetensors pillow

In [ ]:
import torch
from diffusers import QwenImage21Pipeline

pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16,
)
# T4 16GB 装不下整模型：CPU 卸载（能跑、较慢）。如有更大显存可改用 pipe.to("cuda")
# T4 16GB 实测：整块搬运会 OOM，改逐层卸载（2026-11-09 Colab 首跑验证）
pipe.enable_sequential_cpu_offload()
print("模型已加载（CPU offload 模式）")

In [ ]:
import os, time
from datetime import datetime

generator = torch.Generator("cpu").manual_seed(SEED)
t0 = time.time()
image = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT or None,
    height=HEIGHT,
    width=WIDTH,
    num_inference_steps=STEPS,
    true_cfg_scale=1.0,  # 默认不做 CFG；>1 才开启并加倍计算量
    generator=generator,
).images[0]
print("生成用时: %.1fs" % (time.time() - t0))

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
img_path = os.path.join(OUT_DIR, f"qwen-image-2.1_{stamp}_s{SEED}.png")
image.save(img_path)
with open(os.path.join(OUT_DIR, f"qwen-image-2.1_{stamp}_s{SEED}.txt"), "w", encoding="utf-8") as f:
    f.write(f"prompt: {PROMPT}\nnegative_prompt: {NEGATIVE_PROMPT}\nsize: {WIDTH}x{HEIGHT}\nsteps: {STEPS}\nseed: {SEED}\nmodel: Qwen/Qwen-Image-2.1\n")
print("已保存:", img_path)
image

## 首跑检查清单

- 运行时确认是 **T4 GPU**（免费档偶尔分到其他卡，T4 以外也可用，但先按 T4 的参数跑）。
- 首次加载会下载约 30GB 权重，耐心等；若报磁盘空间不足，先清 `/content` 下的旧权重再跑。
- 若 OOM：把 `WIDTH/HEIGHT` 降到 768，或把 `STEPS` 降到 25；仍不行再考虑 int8 量化权重（Comfy-Org 有发布）。
- 生成后去 Drive `free-stack/qwen-image-2.1/` 核对 PNG 与同名 txt（prompt 记录）。